# LeetCode 2320 — Count Number of Ways to Place Houses bằng Python

Link đề: https://leetcode.com/problems/count-number-of-ways-to-place-houses/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def countHousePlacements(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra **hai bên đường độc lập** ⇒ số cách = (số cách một bên)² — **quy tắc nhân**.
2. Đếm dãy nhị phân độ dài `n` **không có hai số 1 kề nhau** bằng DP hai trạng thái (ô cuối có nhà / trống) ⇒ ra **dãy Fibonacci**: `F(n + 2)`.
3. Làm việc **modulo `10⁹ + 7`** đúng chỗ (lấy dư trước khi bình phương vẫn đúng).
4. Tính Fibonacci bằng **lũy thừa ma trận** O(log n) và **fast doubling** — kỹ thuật dùng lại cho mọi truy hồi tuyến tính.
5. Mở rộng: khi hai bên **không** độc lập (cấm nhà đối diện), dùng **ma trận chuyển trạng thái** với 3 trạng thái mỗi cột.

## 2. Đề bài & ví dụ

Một con đường có `n · 2` lô đất, mỗi bên `n` lô đánh số `1 … n`. Mỗi lô có thể đặt một ngôi nhà. Trả về số cách đặt nhà sao cho **không có hai ngôi nhà kề nhau trên cùng một bên** đường, modulo `10⁹ + 7`. Hai lô **đối diện** nhau (cùng số, khác bên) **được phép** cùng có nhà.

**Ví dụ 1:** `n = 1` → `4` (trống cả hai / chỉ bên A / chỉ bên B / cả hai).

**Ví dụ 2:** `n = 2` → `9`.

**Ràng buộc:** `1 ≤ n ≤ 10⁴`.

In [1]:
from itertools import product
import random, time

MOD = 10**9 + 7
examples = [(1, 4), (2, 9)]
for n, exp in examples:
    print(f"n={n} -> kỳ vọng {exp}")

n=1 -> kỳ vọng 4
n=2 -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1 Quy tắc nhân

Nếu một cấu hình gồm hai phần **chọn độc lập** (ràng buộc của phần này không nhắc tới phần kia), số cấu hình = tích số cách của từng phần. Ở đây ràng buộc "không kề nhau cùng bên" chỉ nói về từng bên ⇒ `ans = (số cách một bên)²`.

### 3.2 Dãy nhị phân không có hai số 1 liền nhau

Đặt `a(n)` = số dãy 0/1 độ dài `n` không có `11`. Xét phần tử cuối: là `0` ⇒ phần trước bất kỳ (`a(n−1)` cách); là `1` ⇒ phần tử trước phải là `0`, phần trước nữa bất kỳ (`a(n−2)` cách). ⇒ `a(n) = a(n−1) + a(n−2)`, `a(0) = 1`, `a(1) = 2` ⇒ `a(n) = F(n + 2)` với `F(1) = F(2) = 1`.

In [2]:
def one_side_brute(n):
    return sum(1 for bits in product([0, 1], repeat=n) if all(not (bits[i] and bits[i + 1]) for i in range(n - 1)))

fib = [0, 1]
for _ in range(20): fib.append(fib[-1] + fib[-2])
print("a(n) vét cạn:", [one_side_brute(n) for n in range(1, 11)])
print("F(n+2)      :", [fib[n + 2] for n in range(1, 11)])

a(n) vét cạn: [2, 3, 5, 8, 13, 21, 34, 55, 89, 144]
F(n+2)      : [2, 3, 5, 8, 13, 21, 34, 55, 89, 144]


### 3.3 Modulo và phép nhân

`(x · y) mod p = ((x mod p) · (y mod p)) mod p` ⇒ có thể lấy dư `a(n)` trong suốt DP rồi mới bình phương.

### 3.4 Lũy thừa ma trận cho Fibonacci

`[F(k+1), F(k)]ᵀ = [[1, 1], [1, 0]] · [F(k), F(k−1)]ᵀ` ⇒ `[[1,1],[1,0]]^k = [[F(k+1), F(k)], [F(k), F(k−1)]]`. Lũy thừa nhanh ⇒ O(log k) phép nhân ma trận 2×2.

In [3]:
def mat_mul(A, B, mod=None):
    r = [[A[0][0] * B[0][0] + A[0][1] * B[1][0], A[0][0] * B[0][1] + A[0][1] * B[1][1]],
         [A[1][0] * B[0][0] + A[1][1] * B[1][0], A[1][0] * B[0][1] + A[1][1] * B[1][1]]]
    return [[x % mod for x in row] for row in r] if mod else r

def mat_pow(A, k, mod=None):
    R = [[1, 0], [0, 1]]
    while k:
        if k & 1: R = mat_mul(R, A, mod)
        A = mat_mul(A, A, mod); k >>= 1
    return R

print(mat_pow([[1, 1], [1, 0]], 10), "-> F(11), F(10) =", fib[11], fib[10])

[[89, 55], [55, 34]] -> F(11), F(10) = 89 55


### 3.5 Ma trận chuyển trạng thái = đếm đường đi trong đồ thị trạng thái

Coi mỗi trạng thái của một lô (`0` = trống, `1` = có nhà) là một đỉnh; cạnh `s → t` nếu lô kế tiếp có thể ở trạng thái `t` khi lô hiện tại ở `s`. Cạnh: `0→0`, `0→1`, `1→0` (không có `1→1`). Số dãy độ dài `n` = số **đường đi** độ dài `n − 1` trong đồ thị này = tổng các phần tử của `T^{n−1}` với `T = [[1, 1], [1, 0]]`. Mọi bài "đếm dãy với ràng buộc giữa các phần tử kề nhau" đều theo khuôn này.

In [4]:
T = [[1, 1], [1, 0]]                    # T[s][t]: s -> t hợp lệ (s, t ∈ {0 = trống, 1 = có nhà})
for n in range(1, 8):
    P = mat_pow(T, n - 1)
    print(f"n={n}: tổng phần tử T^{n-1} = {sum(map(sum, P))} = a(n) = {fib[n + 2]}")

n=1: tổng phần tử T^0 = 2 = a(n) = 2
n=2: tổng phần tử T^1 = 3 = a(n) = 3
n=3: tổng phần tử T^2 = 5 = a(n) = 5
n=4: tổng phần tử T^3 = 8 = a(n) = 8
n=5: tổng phần tử T^4 = 13 = a(n) = 13
n=6: tổng phần tử T^5 = 21 = a(n) = 21
n=7: tổng phần tử T^6 = 34 = a(n) = 34


## 4. Ý tưởng nền tảng / trực giác

**Bước 1 — tách hai bên.** Một cách đặt nhà = (cấu hình bên A, cấu hình bên B). Ràng buộc chỉ cấm hai nhà **kề nhau cùng bên** — không có ràng buộc nào giữa A và B. ⇒ `ans = a(n)²`.

**Bước 2 — đếm một bên bằng DP hai trạng thái.**
- `f[i]` = số cách cho `i + 1` lô đầu mà lô thứ `i` **có nhà**;
- `g[i]` = số cách mà lô thứ `i` **trống**.

Chuyển: có nhà ở `i` ⇒ lô `i − 1` phải trống: `f[i] = g[i−1]`. Trống ở `i` ⇒ lô `i − 1` tùy ý: `g[i] = f[i−1] + g[i−1]`. Khởi tạo `f[0] = g[0] = 1`.

Tính tay:

| i | f (có nhà) | g (trống) | a = f + g |
|---|---|---|---|
| 0 (n=1) | 1 | 1 | 2 |
| 1 (n=2) | 1 | 2 | 3 |
| 2 (n=3) | 2 | 3 | 5 |
| 3 (n=4) | 3 | 5 | 8 |

`n = 1`: `2² = 4` ✔; `n = 2`: `3² = 9` ✔.

In [5]:
f = g = 1
for i in range(1, 6):
    f, g = g, f + g
    print(f"n={i + 1}: f={f}, g={g}, một bên = {f + g}, đáp án = {(f + g) ** 2}")

n=2: f=1, g=2, một bên = 3, đáp án = 9
n=3: f=2, g=3, một bên = 5, đáp án = 25
n=4: f=3, g=5, một bên = 8, đáp án = 64
n=5: f=5, g=8, một bên = 13, đáp án = 169
n=6: f=8, g=13, một bên = 21, đáp án = 441


### 4.1 Liệt kê cấu hình một bên — thấy cấu trúc Fibonacci

Mọi cấu hình độ dài `n` hoặc kết thúc bằng `0` (gắn `0` vào một cấu hình độ dài `n − 1`), hoặc kết thúc bằng `1` — khi đó ký tự trước phải là `0`, tức là gắn `01` vào một cấu hình độ dài `n − 2`:

In [6]:
def configs(n):
    if n == 0: return [""]
    if n == 1: return ["0", "1"]
    return [c + "0" for c in configs(n - 1)] + [c + "01" for c in configs(n - 2)]

for n in range(1, 5):
    cs = configs(n)
    assert sorted(cs) == sorted("".join(map(str, b)) for b in product([0, 1], repeat=n) if "11" not in "".join(map(str, b)))
    print(f"n={n}: {len(cs)} cấu hình: {cs}")

n=1: 2 cấu hình: ['0', '1']
n=2: 3 cấu hình: ['00', '10', '01']
n=3: 5 cấu hình: ['000', '100', '010', '001', '101']
n=4: 8 cấu hình: ['0000', '1000', '0100', '0010', '1010', '0001', '1001', '0101']


Cách phân loại theo **đuôi** (`…0` hoặc `…01`) chia tập cấu hình thành hai nhóm rời nhau, cho trực tiếp `a(n) = a(n−1) + a(n−2)` mà không cần hai mảng `f`, `g`.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Duyệt mọi `2^{2n}` cách đặt — O(4ⁿ · n)

In [7]:
def brute(n):
    cnt = 0
    for A in product([0, 1], repeat=n):
        for B in product([0, 1], repeat=n):
            if all(not (A[i] and A[i + 1]) and not (B[i] and B[i + 1]) for i in range(n - 1)):
                cnt += 1
    return cnt % MOD

for n, exp in examples:
    print(brute(n), "OK" if brute(n) == exp else "SAI")

4 OK
9 OK


`n = 10⁴` ⇒ `4^{10⁴}` — bất khả thi; dùng kiểm chứng tới `n ≈ 8`.

### 5.2 Baseline 2 — DP trên **cả hai bên cùng lúc** (4 trạng thái mỗi cột) — không cần nhận xét độc lập

Trạng thái cột `i` là cặp `(A_i, B_i) ∈ {0,1}²`; chuyển hợp lệ nếu không có `A_{i−1} = A_i = 1` hay `B_{i−1} = B_i = 1`. Kết quả phải trùng `a(n)²` — một cách kiểm chứng quy tắc nhân.

In [8]:
def dp_four_states(n):
    states = [(x, y) for x in (0, 1) for y in (0, 1)]
    cur = {s: 1 for s in states}
    for _ in range(n - 1):
        nxt = {s: 0 for s in states}
        for (pa, pb), w in cur.items():
            for (a, b) in states:
                if not (pa and a) and not (pb and b):
                    nxt[(a, b)] = (nxt[(a, b)] + w) % MOD
        cur = nxt
    return sum(cur.values()) % MOD

for n, exp in examples:
    print(dp_four_states(n), "OK" if dp_four_states(n) == exp else "SAI")
print([dp_four_states(n) for n in range(1, 8)], "vs a(n)^2:", [fib[n + 2] ** 2 for n in range(1, 8)])

4 OK
9 OK
[4, 9, 25, 64, 169, 441, 1156] vs a(n)^2: [4, 9, 25, 64, 169, 441, 1156]


**Nút thắt:** DP 4 trạng thái đúng nhưng làm việc thừa — quy tắc nhân cho phép đếm một bên (2 trạng thái) rồi bình phương.

## 6. Ý tưởng thuật toán / bước đột phá

```
f, g = 1, 1                      # lô đầu: có nhà / trống
for i in 2..n: f, g = g, (f + g) % MOD
one = (f + g) % MOD              # = F(n + 2) mod p
return one * one % MOD
```

- O(n) thời gian, O(1) bộ nhớ.
- Có thể tính `F(n + 2)` trong O(log n) bằng lũy thừa ma trận hoặc fast doubling — với `n ≤ 10⁴` không cần, nhưng là kỹ thuật chuẩn khi `n` tới `10¹⁸`.

### 6.1 Lấy dư từng bước hay để số lớn rồi mới lấy dư?

Kết quả như nhau (tính chất đồng dư), nhưng không lấy dư thì số `F(n + 2)` có hàng nghìn chữ số và mỗi phép cộng tốn O(số chữ số) ⇒ tổng O(n²) thao tác bit. Với `n = 10⁴` vẫn nhanh, nhưng chênh lệch thấy rõ khi `n` tăng:

In [9]:
for n in (10**4, 5 * 10**4):
    t0 = time.perf_counter(); a, b = 1, 1
    for _ in range(n - 1): a, b = b, a + b
    big = (a + b) ** 2 % MOD; t_big = time.perf_counter() - t0
    t0 = time.perf_counter(); f = g = 1
    for _ in range(n - 1): f, g = g, (f + g) % MOD
    small = (f + g) * (f + g) % MOD; t_mod = time.perf_counter() - t0
    assert big == small
    print(f"n={n}: không lấy dư {t_big*1000:7.1f} ms | lấy dư mỗi bước {t_mod*1000:6.1f} ms")

n=10000: không lấy dư     2.0 ms | lấy dư mỗi bước    1.0 ms
n=50000: không lấy dư    42.7 ms | lấy dư mỗi bước    5.9 ms


## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP hai mảng `f`, `g` (cách doocs)

In [10]:
def dp_arrays(n):
    f = [1] * n; g = [1] * n
    for i in range(1, n):
        f[i] = g[i - 1]
        g[i] = (f[i - 1] + g[i - 1]) % MOD
    v = f[-1] + g[-1]
    return v * v % MOD

for n, exp in examples:
    print(dp_arrays(n), "OK" if dp_arrays(n) == exp else "SAI")

4 OK
9 OK


### 7.2 Cách 3 — DP hai biến (O(1) bộ nhớ)

In [11]:
def dp_rolling(n):
    f = g = 1
    for _ in range(n - 1):
        f, g = g, (f + g) % MOD
    one = (f + g) % MOD
    return one * one % MOD

for n, exp in examples:
    print(dp_rolling(n), "OK" if dp_rolling(n) == exp else "SAI")

4 OK
9 OK


### 7.3 Cách 4 — Lũy thừa ma trận: `F(n + 2)` trong O(log n)

In [12]:
def matrix_power(n):
    M_ = mat_pow([[1, 1], [1, 0]], n + 1, MOD)       # [[F(n+2), F(n+1)], [F(n+1), F(n)]]
    one = M_[0][0]
    return one * one % MOD

for n, exp in examples:
    print(matrix_power(n), "OK" if matrix_power(n) == exp else "SAI")

4 OK
9 OK


### 7.4 Cách 5 — Fast doubling: `F(2k) = F(k)·(2F(k+1) − F(k))`, `F(2k+1) = F(k)² + F(k+1)²`

In [13]:
def fib_pair(k):                                     # trả (F(k), F(k+1)) mod MOD
    if k == 0: return (0, 1)
    a, b = fib_pair(k >> 1)
    c = a * ((2 * b - a) % MOD) % MOD
    d = (a * a + b * b) % MOD
    return (d, (c + d) % MOD) if k & 1 else (c, d)

def fast_doubling(n):
    one = fib_pair(n + 2)[0]
    return one * one % MOD

for n, exp in examples:
    print(fast_doubling(n), "OK" if fast_doubling(n) == exp else "SAI")
print("F(10..15):", [fib_pair(k)[0] for k in range(10, 16)], "| tham chiếu:", fib[10:16])

4 OK
9 OK
F(10..15): [55, 89, 144, 233, 377, 610] | tham chiếu: [55, 89, 144, 233, 377, 610]


Độ sâu đệ quy chỉ `log₂ n` (≈ 14 với `n = 10⁴`; ≈ 60 với `10¹⁸`) — an toàn.

### 7.5 Cách 6 — Tổ hợp: số dãy có đúng `j` nhà là `C(n − j + 1, j)`

Đặt `j` nhà không kề nhau vào `n` lô ⇔ chọn `j` vị trí trong `n − j + 1` "khe" (tư duy "chia kẹo"). ⇒ `a(n) = Σ_j C(n − j + 1, j)`.

In [14]:
from math import comb
def combinatorial(n):
    one = sum(comb(n - j + 1, j) for j in range(0, (n + 1) // 2 + 1)) % MOD
    return one * one % MOD

for n, exp in examples:
    print(combinatorial(n), "OK" if combinatorial(n) == exp else "SAI")
print([sum(comb(n - j + 1, j) for j in range(0, (n + 1) // 2 + 1)) for n in range(1, 11)], "= F(n+2)")

4 OK
9 OK
[2, 3, 5, 8, 13, 21, 34, 55, 89, 144] = F(n+2)


## 8. TRACE từng bước

### 8.1 Liệt kê 9 cách của `n = 2`

In [15]:
one_side = ["".join(map(str, b)) for b in product([0, 1], repeat=2) if not (b[0] and b[1])]
print("một bên hợp lệ:", one_side)
for k, (A, B) in enumerate(product(one_side, repeat=2), 1):
    print(f"{k}. bên A = {A}, bên B = {B}")

một bên hợp lệ: ['00', '01', '10']
1. bên A = 00, bên B = 00
2. bên A = 00, bên B = 01
3. bên A = 00, bên B = 10
4. bên A = 01, bên B = 00
5. bên A = 01, bên B = 01
6. bên A = 01, bên B = 10
7. bên A = 10, bên B = 00
8. bên A = 10, bên B = 01
9. bên A = 10, bên B = 10


Mỗi bên có 3 cấu hình `00, 01, 10` (không được `11`) ⇒ `3 × 3 = 9`. Cấu hình `A = 10, B = 10` có hai nhà **đối diện** nhau — hợp lệ theo đề.

### 8.2 Trace DP lăn với modulo cho `n` lớn

In [16]:
f = g = 1
for i in range(2, 31):
    f, g = g, (f + g) % MOD
    if i in (5, 10, 20, 30):
        one = (f + g) % MOD
        print(f"n={i}: một bên = {one} (F({i + 2}) = {fib_pair(i + 2)[0]}), đáp án = {one * one % MOD}")

n=5: một bên = 13 (F(7) = 13), đáp án = 169
n=10: một bên = 144 (F(12) = 144), đáp án = 20736
n=20: một bên = 17711 (F(22) = 17711), đáp án = 313679521
n=30: một bên = 2178309 (F(32) = 2178309), đáp án = 30066266


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def countHousePlacements(self, n: int) -> int:
        MOD = 10**9 + 7
        occupied, empty = 1, 1                 # lô đầu tiên: có nhà / trống
        for _ in range(n - 1):
            occupied, empty = empty, (occupied + empty) % MOD   # có nhà <= trước trống; trống <= tùy ý
        one_side = (occupied + empty) % MOD    # = F(n + 2)
        return one_side * one_side % MOD       # hai bên đường độc lập

In [18]:
sol = Solution()
for n, exp in examples:
    got = sol.countHousePlacements(n)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

4 | kỳ vọng 4 | OK
9 | kỳ vọng 9 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (độc lập).** Tập cấu hình hợp lệ = `S × S` với `S` = tập dãy 0/1 độ dài `n` không có `11`. *Chứng minh:* điều kiện hợp lệ là hội của "bên A không có `11`" và "bên B không có `11`"; mỗi điều kiện chỉ phụ thuộc một bên. ⇒ `|S × S| = |S|²`.

**Bổ đề 2 (DP một bên).** Đặt `f_i`, `g_i` = số dãy hợp lệ độ dài `i + 1` kết thúc bằng 1 / 0. Dãy kết thúc bằng 1 hợp lệ ⇔ phần đầu (dài `i`) hợp lệ và kết thúc bằng 0 ⇒ `f_i = g_{i−1}`. Dãy kết thúc bằng 0 hợp lệ ⇔ phần đầu hợp lệ (kết thúc bằng gì cũng được) ⇒ `g_i = f_{i−1} + g_{i−1}`. `f_0 = g_0 = 1`. ⇒ `|S| = f_{n−1} + g_{n−1}`.

**Bổ đề 3 (Fibonacci).** `a_i = f_i + g_i = g_{i−1} + f_{i−1} + g_{i−1} = a_{i−1} + g_{i−1} = a_{i−1} + a_{i−2}` (vì `g_{i−1} = a_{i−2}`). Với `a_0 = 2 = F(3)`, `a_1 = 3 = F(4)` ⇒ `a_{n−1} = F(n + 2)`.

**Modulo.** Mọi phép cộng/nhân trong DP và bình phương được lấy dư, đúng theo tính chất đồng dư. ∎

**Công thức tổ hợp (cách 6).** Đặt `j` nhà không kề nhau vào `n` lô: bỏ đi `j − 1` lô trống bắt buộc nằm giữa các nhà liên tiếp, ta được bài toán chọn `j` vị trí **bất kỳ** trong `n − (j − 1) = n − j + 1` vị trí — song ánh giữa hai tập ⇒ `C(n − j + 1, j)` cách. Cộng theo mọi `j` cho `a(n)` — một chứng minh tổ hợp của đẳng thức `Σ_j C(n − j + 1, j) = F(n + 2)` (tổng theo "đường chéo nông" của tam giác Pascal).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `4ⁿ` | O(4ⁿ · n) | O(n) |
| 1b. DP 4 trạng thái | O(16 n) | O(1) |
| 2. DP hai mảng | O(n) | O(n) |
| 3. DP hai biến | O(n) | O(1) |
| 4. Lũy thừa ma trận | O(log n) | O(1) |
| 5. Fast doubling | O(log n) | O(log n) đệ quy |
| 6. Tổ hợp | O(n²) với `comb` lớn | O(1) |

In [19]:
res = []
for name, f in [("dp_four_states", dp_four_states), ("dp_arrays", dp_arrays), ("dp_rolling", dp_rolling),
                ("matrix_power", matrix_power), ("fast_doubling", fast_doubling), ("Solution", Solution().countHousePlacements)]:
    t0 = time.perf_counter(); res.append(f(10**4)); dt = time.perf_counter() - t0
    print(f"{name:<15} n=10^4: {dt*1000:7.2f} ms -> {res[-1]}")
assert len(set(res)) == 1
t0 = time.perf_counter(); v = fast_doubling(10**18); print(f"fast_doubling n=10^18: {(time.perf_counter()-t0)*1000:.2f} ms -> {v}")
assert matrix_power(10**18) == v

dp_four_states  n=10^4:   16.78 ms -> 402613600
dp_arrays       n=10^4:    1.32 ms -> 402613600
dp_rolling      n=10^4:    0.53 ms -> 402613600
matrix_power    n=10^4:    0.02 ms -> 402613600
fast_doubling   n=10^4:    0.01 ms -> 402613600
Solution        n=10^4:    0.53 ms -> 402613600
fast_doubling n=10^18: 0.07 ms -> 6323684


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "dp_four_states": dp_four_states, "dp_arrays": dp_arrays, "dp_rolling": dp_rolling,
       "matrix_power": matrix_power, "fast_doubling": fast_doubling, "combinatorial": combinatorial,
       "Solution": Solution().countHousePlacements}
edge = examples + [(3, 25), (4, 64), (5, 169), (6, 441)]
for n, exp in edge:
    for name, f in ALL.items():
        got = f(n)
        assert got == exp, (name, n, got, exp)
    print(f"n={n} -> {exp} = {int(exp ** 0.5)}^2")
print("edge OK")

n=1 -> 4 = 2^2
n=2 -> 9 = 3^2
n=3 -> 25 = 5^2
n=4 -> 64 = 8^2
n=5 -> 169 = 13^2
n=6 -> 441 = 21^2
edge OK


### 12.2 Đối chiếu vét cạn với `n ≤ 8` và giữa các cách với `n` ngẫu nhiên lớn

In [21]:
for n in range(1, 9):
    assert brute(n) == Solution().countHousePlacements(n) == dp_four_states(n)
rng = random.Random(42)
for _ in range(200):
    n = rng.randint(1, 10**4)
    ref = dp_rolling(n)
    assert matrix_power(n) == fast_doubling(n) == Solution().countHousePlacements(n) == ref
print("đối chiếu OK")

đối chiếu OK


### 12.3 Tính chất: đáp án là số chính phương (trước khi lấy dư, với n nhỏ); tỉ số liên tiếp → φ² ≈ 2,618

In [22]:
vals = [fib[n + 2] ** 2 for n in range(1, 18)]
assert all(Solution().countHousePlacements(n) == v % MOD for n, v in zip(range(1, 18), vals))
print("tỉ số:", [round(vals[i + 1] / vals[i], 4) for i in range(10, 16)], "| φ² =", round(((1 + 5 ** 0.5) / 2) ** 2, 4))

tỉ số: [2.618, 2.618, 2.618, 2.618, 2.618, 2.618] | φ² = 2.618


Số cách tăng theo hàm mũ `≈ φ^{2n}` ⇒ với `n = 10⁴` có khoảng `4180` chữ số thập phân trước khi lấy dư — đó là lý do đề yêu cầu modulo.

### 12.4 Edge cực trị: `n = 10⁴` không lấy dư trong DP (số lớn Python) vẫn cho cùng kết quả sau khi lấy dư

In [23]:
a, b = 1, 1
for _ in range(10**4 - 1): a, b = b, a + b
one = a + b
print("số chữ số của một bên:", len(str(one)) if one.bit_length() < 13000 else "rất lớn", "| bit_length:", one.bit_length())
assert one * one % MOD == Solution().countHousePlacements(10**4)

số chữ số của một bên: 2090 | bit_length: 6943


### 12.5 Kiểm chứng các đẳng thức fast doubling trên số nguyên chính xác

Hai đẳng thức `F(2k) = F(k)·(2F(k+1) − F(k))` và `F(2k+1) = F(k)² + F(k+1)²` suy ra từ việc bình phương ma trận `[[1,1],[1,0]]^k`. Kiểm tra trực tiếp với số nguyên lớn (không lấy dư) cho `k` tới 300:

In [24]:
F = [0, 1]
for _ in range(700): F.append(F[-1] + F[-2])
for k in range(1, 300):
    assert F[2 * k] == F[k] * (2 * F[k + 1] - F[k])
    assert F[2 * k + 1] == F[k] ** 2 + F[k + 1] ** 2
print("hai đẳng thức fast doubling đúng với k = 1..299")

hai đẳng thức fast doubling đúng với k = 1..299


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm **một bên** đường

In [25]:
def wrong_one_side(n):
    f = g = 1
    for _ in range(n - 1): f, g = g, (f + g) % MOD
    return (f + g) % MOD                               # SAI: quên bên kia

for n, _ in examples:
    print("bản sai :", wrong_one_side(n), "| bản đúng:", Solution().countHousePlacements(n))

bản sai : 2 | bản đúng: 4
bản sai : 3 | bản đúng: 9


**Sửa:** bình phương (quy tắc nhân).

### Lỗi 2 — **Cộng** hai bên thay vì nhân

In [26]:
def wrong_add(n):
    one = wrong_one_side(n)
    return 2 * one % MOD                               # SAI: các cấu hình hai bên kết hợp tự do => nhân

for n, _ in examples:
    print("bản sai :", wrong_add(n), "| bản đúng:", Solution().countHousePlacements(n))

bản sai : 4 | bản đúng: 4
bản sai : 6 | bản đúng: 9


**Vì sao sai:** `n = 1` **trùng hợp** đúng (`2 + 2 = 2 · 2 = 4`). `n = 2`: `3 + 3 = 6 ≠ 9`. Cộng dùng khi chọn **một trong hai** lựa chọn loại trừ nhau; nhân dùng khi chọn **cả hai** độc lập.

### Lỗi 3 — Cấm luôn hai nhà **đối diện** nhau (đọc sai đề)

In [27]:
def wrong_no_opposite(n):
    # 3 trạng thái mỗi cột: (0,0), (1,0), (0,1) — cấm (1,1)
    states = [(0, 0), (1, 0), (0, 1)]
    cur = {s: 1 for s in states}
    for _ in range(n - 1):
        nxt = {s: 0 for s in states}
        for (pa, pb), w in cur.items():
            for (a, b) in states:
                if not (pa and a) and not (pb and b): nxt[(a, b)] += w
        cur = nxt
    return sum(cur.values()) % MOD                     # SAI cho bài gốc (đúng cho biến thể 15.1)

for n, _ in examples:
    print("bản sai :", wrong_no_opposite(n), "| bản đúng:", Solution().countHousePlacements(n))

bản sai : 3 | bản đúng: 4
bản sai : 7 | bản đúng: 9


**Vì sao sai:** đề nói rõ hai lô đối diện **được** cùng có nhà (`n = 1` có 4 cách gồm "hai nhà, mỗi bên một"). **Sửa:** không có ràng buộc giữa hai bên.

### Lỗi 4 — Lệch chỉ số Fibonacci (`F(n + 1)` thay vì `F(n + 2)`)

In [28]:
def wrong_index(n):
    one = fib_pair(n + 1)[0]                           # SAI: lệch một
    return one * one % MOD

for n, _ in examples:
    print("bản sai :", wrong_index(n), "| bản đúng:", Solution().countHousePlacements(n))

bản sai : 1 | bản đúng: 4
bản sai : 4 | bản đúng: 9


**Vì sao sai:** `a(1) = 2` (trống / có nhà) = `F(3)`, không phải `F(2) = 1`. Bản sai cho `1, 4, 9, 25, …` — chính là đáp án đúng **dịch đi một bước** (`n = 2` cho 4 là đáp án của `n = 1`), kiểu lỗi rất khó thấy nếu chỉ thử một giá trị. Luôn kiểm tra công thức đóng với vài `n` nhỏ bằng vét cạn.

### Lỗi 5 — Khởi tạo DP sai: chỉ tính "trống" cho lô đầu

In [29]:
def wrong_init(n):
    f, g = 0, 1                                        # SAI: lô đầu có nhà cũng là 1 cách
    for _ in range(n - 1): f, g = g, (f + g) % MOD
    one = (f + g) % MOD
    return one * one % MOD

for n, _ in examples:
    print("bản sai :", wrong_init(n), "| bản đúng:", Solution().countHousePlacements(n))

bản sai : 1 | bản đúng: 4
bản sai : 4 | bản đúng: 9


**Vì sao sai:** khởi tạo `f = 0` nghĩa là "lô đầu không được có nhà" — cũng làm dãy lệch một bước giống Lỗi 4 (ở đây hai lỗi khác nguyên nhân nhưng cho cùng output). **Sửa:** `f₀ = g₀ = 1` (lô đầu có nhà: 1 cách; trống: 1 cách). Hoặc khởi tạo từ "trạng thái rỗng" `f = 0, g = 1` và chạy `n` bước (không phải `n − 1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc chỉ trên từng phần ⇒ **quy tắc nhân**; luôn kiểm tra xem có ràng buộc "chéo" nào không.
- "Không có hai phần tử liền nhau cùng chọn" ⇒ Fibonacci; DP hai trạng thái (chọn / không chọn phần tử cuối).
- Truy hồi tuyến tính cấp `k` với `n` cực lớn ⇒ **lũy thừa ma trận** `k × k`.
- Modulo: lấy dư sau mỗi phép cộng/nhân; Python không tràn số nhưng số lớn làm chậm.
- Thử công thức với **nhiều** giá trị `n` nhỏ (1…6) bằng vét cạn: lỗi lệch chỉ số thường cho ra đúng dãy đáp án nhưng **trượt một vị trí**, chỉ một điểm kiểm tra không phát hiện được.
- Khi đề cho ví dụ có hình, đọc kỹ chú thích về các ràng buộc "chéo" (ở đây: nhà đối diện **được phép**) — chi tiết đó quyết định có dùng được quy tắc nhân hay không.
- Bài cùng họ: 70 Climbing Stairs, 198 House Robber, 509 Fibonacci Number, 1137 N-th Tribonacci, 1411 Number of Ways to Paint N × 3 Grid, 2466 Count Ways To Build Good Strings, 790 Domino and Tromino Tiling.

## 15. Biến thể của bài

### 15.1 Biến thể A — **Cấm** cả hai nhà đối diện (hai bên không còn độc lập)

3 trạng thái mỗi cột `(0,0), (1,0), (0,1)`; ma trận chuyển `T` 3 × 3; đáp án `= 1ᵀ · T^{n−1} · 1`. Dãy kết quả `3, 7, 17, 41, …` thỏa `a(n) = 2a(n−1) + a(n−2)` (số Pell).

In [30]:
def no_opposite_brute(n):
    cnt = 0
    for A in product([0, 1], repeat=n):
        for B in product([0, 1], repeat=n):
            if all(not (A[i] and B[i]) for i in range(n)) and \
               all(not (A[i] and A[i + 1]) and not (B[i] and B[i + 1]) for i in range(n - 1)):
                cnt += 1
    return cnt

def no_opposite_pell(n):
    a, b = 1, 3                                        # a(0) = 1, a(1) = 3
    for _ in range(n - 1): a, b = b, (2 * b + a) % MOD
    return b

for n in range(1, 8):
    assert no_opposite_brute(n) == wrong_no_opposite(n) == no_opposite_pell(n)
print([no_opposite_pell(n) for n in range(1, 9)])

[3, 7, 17, 41, 99, 239, 577, 1393]


### 15.2 Biến thể B — `k` dãy song song độc lập: `a(n)^k`

In [31]:
def k_rows(n, k):
    return pow(fib_pair(n + 2)[0], k, MOD)
print(k_rows(1, 2), k_rows(2, 2), k_rows(2, 3), "| kiểm: 3^3 =", 27)

4 9 27 | kiểm: 3^3 = 27


### 15.3 Biến thể C — Giữa hai nhà cùng bên phải cách **ít nhất `d`** lô trống

Truy hồi `a(n) = a(n − 1) + a(n − d − 1)` (lô cuối trống, hoặc có nhà và `d` lô trước trống), với `a(m) = m + 1` cho `0 ≤ m ≤ d`… — kiểm chứng bằng vét cạn:

In [32]:
def gap_at_least(n, d):
    a = [0] * (n + 1)
    for m in range(n + 1):
        a[m] = (m + 1) if m <= d else (a[m - 1] + a[m - d - 1]) % MOD
    return a[n] * a[n] % MOD

def gap_brute(n, d):
    def ok(b):
        last = None
        for i, x in enumerate(b):
            if x:
                if last is not None and i - last <= d: return False
                last = i
        return True
    one = sum(1 for b in product([0, 1], repeat=n) if ok(b))
    return one * one % MOD

for d in range(1, 4):
    for n in range(1, 11):
        assert gap_at_least(n, d) == gap_brute(n, d), (n, d)
print([gap_at_least(n, 2) for n in range(1, 9)], "(d = 2)")

[4, 9, 16, 36, 81, 169, 361, 784] (d = 2)


Với `d = 1` ta có lại bài gốc. Khi `d` tăng, số cách một bên tăng chậm hơn (tốc độ là nghiệm lớn nhất của `x^{d+1} = x^d + 1`), ví dụ `d = 2` cho dãy Narayana's cows `1, 2, 3, 4, 6, 9, 13, …`. Với `m ≤ d` chỉ đặt được tối đa một nhà ⇒ `m + 1` cách (trống hết, hoặc một nhà ở một trong `m` lô).

### 15.4 Biến thể D — Đường **vòng** (lô 1 kề lô `n`)

Số dãy vòng không có hai số 1 kề nhau là **số Lucas** `L(n) = F(n − 1) + F(n + 1)` (với `n ≥ 2`).

In [33]:
def circular_one_side_brute(n):
    return sum(1 for b in product([0, 1], repeat=n) if all(not (b[i] and b[(i + 1) % n]) for i in range(n)))

def circular(n):
    one = (fib_pair(n - 1)[0] + fib_pair(n + 1)[0]) % MOD if n >= 2 else 2
    return one * one % MOD

for n in range(2, 12):
    assert circular(n) == circular_one_side_brute(n) ** 2 % MOD
print([circular_one_side_brute(n) for n in range(2, 10)], "= Lucas")

[3, 4, 7, 11, 18, 29, 47, 76] = Lucas


### 15.5 Biến thể E — Đếm số cách có **tổng đúng `m` nhà** (cả hai bên)

Gọi `c_j = C(n − j + 1, j)` = số cách một bên có đúng `j` nhà. Hai bên độc lập ⇒ số cách có tổng `m` nhà là **tích chập** `Σ_j c_j · c_{m−j}`.

In [34]:
def exactly_m_total(n, m):
    c = [comb(n - j + 1, j) for j in range(n + 1)]
    return sum(c[j] * c[m - j] for j in range(max(0, m - n), min(m, n) + 1)) % MOD

def exactly_m_brute(n, m):
    one = [b for b in product([0, 1], repeat=n) if all(not (b[i] and b[i + 1]) for i in range(n - 1))]
    return sum(1 for A in one for B in one if sum(A) + sum(B) == m)

for n in range(1, 7):
    for m in range(0, 2 * n + 1):
        assert exactly_m_total(n, m) == exactly_m_brute(n, m)
    assert sum(exactly_m_total(n, m) for m in range(2 * n + 1)) == Solution().countHousePlacements(n)
print([exactly_m_total(3, m) for m in range(7)], "-> tổng", sum(exactly_m_total(3, m) for m in range(7)))

[1, 6, 11, 6, 1, 0, 0] -> tổng 25


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | định nghĩa | O(4ⁿ n) | O(n) | kiểm chứng |
| 1b. DP 4 trạng thái | không cần độc lập | O(n) | O(1) | khi có ràng buộc chéo |
| 2–3. DP một bên + bình phương | quy tắc nhân | O(n) | O(1) | **mặc định** |
| 4. Lũy thừa ma trận | truy hồi tuyến tính | O(log n) | O(1) | `n` cực lớn |
| 5. Fast doubling | công thức nhân đôi | O(log n) | O(log n) | `n` cực lớn, nhanh nhất |
| 6. Tổ hợp | đếm theo số nhà | O(n²) | O(1) | khi cần số cách có đúng `j` nhà |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countHousePlacements(3)` và `countHousePlacements(7)`?

**Bài 2 (tính tay):** Lập bảng `f, g` tới `n = 6` và tính đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(n, j)` trả về số cách đặt **đúng `j` nhà trên một bên** (không kề nhau), dùng DP (không dùng `comb`).

**Bài 4 (tự viết lại):** Viết `bt4_todo(n)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(n)` trả về số cách cho biến thể **cấm nhà đối diện** bằng lũy thừa ma trận 3 × 3 (O(log n)).

In [35]:
def bt3_todo(n, j):
    # TODO: đúng j nhà trên một bên
    pass

def bt4_todo(n):
    # TODO: bản nộp
    pass

def bt5_todo(n):
    # TODO: ma trận 3x3 cho biến thể cấm đối diện
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `a(3) = F(5) = 5`. Mức 2: `25`. Mức 3: `a(7) = F(9) = 34` ⇒ `1156`.

**Bài 2.** Mức 1: `(f, g) = (1,1), (1,2), (2,3), (3,5), (5,8), (8,13)`. Mức 2: `a(6) = 21`. Mức 3: `441`.

**Bài 3.** Mức 1: thêm chiều "số nhà đã đặt". Mức 2: `F[i][c]`, `G[i][c]` như cũ nhưng `F[i][c] = G[i−1][c−1]`. Mức 3: đáp án `F[n−1][j] + G[n−1][j]`; kiểm tra bằng `C(n − j + 1, j)`.

**Bài 4.** Mức 1: hai biến. Mức 2: `n − 1` bước. Mức 3: bình phương, lấy dư.

**Bài 5.** Mức 1: `T[s][t] = 1` nếu chuyển `s → t` hợp lệ. Mức 2: lũy thừa `T^{n−1}`. Mức 3: tổng mọi phần tử của `T^{n−1}` (khởi đầu mọi trạng thái có trọng số 1).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `25`, `1156`. **Bài 2:** `a(6) = 21`, đáp án `441`.

In [36]:
assert Solution().countHousePlacements(3) == 25 and Solution().countHousePlacements(7) == 1156
assert Solution().countHousePlacements(6) == 441

def bt3(n, j):
    F = [0] * (j + 1); G = [0] * (j + 1)
    G[0] = 1
    if j >= 1: F[1] = 1
    for _ in range(n - 1):
        nF = [0] + [G[c - 1] for c in range(1, j + 1)]
        nG = [(F[c] + G[c]) % MOD for c in range(j + 1)]
        F, G = nF, nG
    return (F[j] + G[j]) % MOD

def bt4(n):
    return dp_rolling(n)

def bt5(n):
    states = [(0, 0), (1, 0), (0, 1)]
    T = [[1 if not (s[0] and t[0]) and not (s[1] and t[1]) else 0 for t in states] for s in states]
    def mul(A, B):
        return [[sum(A[i][k] * B[k][j] for k in range(3)) % MOD for j in range(3)] for i in range(3)]
    R = [[int(i == j) for j in range(3)] for i in range(3)]; P = T; k = n - 1
    while k:
        if k & 1: R = mul(R, P)
        P = mul(P, P); k >>= 1
    return sum(map(sum, R)) % MOD

for n in range(1, 12):
    for j in range(0, n + 1):
        assert bt3(n, j) == comb(n - j + 1, j) % MOD
    assert bt4(n) == brute(n) if n <= 7 else bt4(n) == dp_rolling(n)
    assert bt5(n) == no_opposite_pell(n)
assert bt5(60) == no_opposite_pell(60)
big = bt5(10**12); assert big == (2 * bt5(10**12 - 1) + bt5(10**12 - 2)) % MOD   # truy hồi Pell ở n rất lớn
print([bt3(6, j) for j in range(4)], bt5(5), no_opposite_pell(5))
print("Đáp án bài tập: tất cả assert OK")

[1, 6, 10, 4] 99 99
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
MOD = 10**9 + 7
f = g = 1                                   # lô đầu: có nhà / trống
for _ in range(n - 1):
    f, g = g, (f + g) % MOD                 # có nhà <- trước trống; trống <- tùy ý
one = (f + g) % MOD                         # = F(n+2)
return one * one % MOD                      # hai bên độc lập
```

**Checklist:**
- [ ] Bình phương (hai bên độc lập), không cộng.
- [ ] Hai nhà **đối diện** được phép.
- [ ] Khởi tạo `f = g = 1`, chạy `n − 1` bước (hoặc `F(n + 2)`).
- [ ] Lấy dư mỗi bước.